In [1]:
import pandas as pd

#read the dataset
data = pd.read_csv("sales.csv", encoding_errors="replace")
data['date'] = pd.to_datetime(data['date'], format='ISO8601')#yearfirst=True)

In [2]:
data.info(memory_usage='deep') #.head()

In [3]:
data

In [4]:

data.iloc[:,1:3].nunique().rename('number of unique values')#.reset_index(name='number of unique values')#.index

data_cols = data.iloc[:,1:3].columns
for col in data_cols:
    print(f"{col}\t", data[col].unique())

In [5]:
# To check if all days are present in a Time Series
def check_missing_days(ts):
    import pandas as pd

    # Example time series
    # dates = pd.date_range("2023-01-01", "2023-01-10").delete([3, 7])  # missing Jan 4 and Jan 8
    # ts = pd.Series(range(len(dates)), index=dates)
    # ts.index.values[0]='2022-12-29'
    # ts[2]='2022-12-21'

    # Expected full range
    full_range = pd.date_range(ts.min(), ts.max(), freq="D")
    # full_range = pd.date_range(ts.index.min(), ts.index.max(), freq="D")

    # Check if all days are present
    missing_days = full_range.difference(ts)
    print(missing_days)

check_missing_days(data.copy()['date'])

In [6]:
data['month'] = data['date'].dt.to_period('M').astype(str)
data[['month', 'date']]#.info()

## Product Sales Leaderboard

In [ ]:
product_total_sales = (
    data.groupby('product')
        .agg(total_sales=('sales', 'sum'))
        # .reset_index()
        .sort_values('total_sales', ascending=False)
)
product_total_sales.index.name=None

product_total_sales.plot.bar(#'product',
                             #'total_sales', 
                             title='Product Sales Leaderboard', 
                             # xlabel='Product Name',
                             # ylabel='Total Sales',
                             grid=True,
                             rot=0)

## Regional Sales Comparison

In [ ]:
region_total_sales = (data.groupby('region')
                          .agg(total_sales=('sales', 'sum'))
                          .reset_index()
                          .sort_values('total_sales', ascending=False))

region_total_sales.plot.bar('region', 'total_sales', 
                             title='Region Sales Comparison', 
                             xlabel='Region',
                             ylabel='Total Sales',
                             rot=0)

## Monthly Trend Analysis

In [ ]:
monthly_total_sales = (data.groupby('month')
                          .agg(total_sales=('sales', 'sum'))
                          .reset_index()
                      )

monthly_total_sales.plot.line('month', 'total_sales', 
                             title='Monthly Trend Analysis', 
                             xlabel='Month',
                             ylabel='Total Sales',
                             grid=True,
                             rot=0)

## Units Sold vs Revenue

In [ ]:
product_total_units_sold = (
            data.groupby('product')
                .agg(total_sales=('sales', 'sum'),
                     total_units_sold=('units_sold', 'sum'))
                .reset_index()
                .sort_values('total_units_sold', ascending=False)
)
product_total_units_sold

In [ ]:
product_total_units_sold.set_index('product', inplace=True)
product_total_units_sold.index.name=None
product_total_units_sold.plot.bar(
                             title='Units Sold vs Revenue', 
                             # xlabel='Product Name',
                             # ylabel='Total Sales',
                             grid=True,
                             rot=0)


## Average Order Value (AOV)

In [ ]:
data['AOV'] = data['sales'] / data['units_sold']
product_total_AOV = (
            data.groupby('region')
                .agg(AverageOrderValue=('AOV', 'mean'))
                .sort_values('AverageOrderValue', ascending=False)
                .rename(columns={'AverageOrderValue':'Average Order Value'})
)
product_total_AOV.index.name=None
product_total_AOV.plot.bar(
             title='Region Average Order Value',
             grid=True,
             rot=0
)

In [ ]:
data['AOV'] = data['sales'] / data['units_sold']
product_total_AOV = (
    data.groupby('product')['AOV'].mean()
        .reset_index(name='Average Order Value')
        .sort_values('Average Order Value', ascending=False)
)
product_total_AOV.set_index('product', inplace=True)
product_total_AOV.plot.bar(
             title='Product Average Order Value',
             xlabel='',
             # ylabel='Total Sales',
             grid=True,
             rot=0
)
# product_total_AOV.index